# Capstone A — Wavely Podcasts

**Level:** getting started

Wavely recently added podcasts. The content team asks:
1. Which **shows** get the most listening hours?
2. Which **categories** do people actually finish? (**completion rate** = seconds listened / episode duration)
3. How does podcast listening develop **per day and device**?

Build a small, trustworthy pipeline that answers these questions and present it in session 10.

## The data (`data/capstone/podcasts/`)
| File | Columns |
|---|---|
| `shows.csv` | `show_id, show_name, category, language` |
| `episodes.csv` | `episode_id, show_id, title, duration_sec, published_date` |
| `listens_2026-09-01.csv` … `listens_2026-09-07.csv` | `listen_id, listened_at, user_id, episode_id, seconds_listened, device` |

The listen files are **not** perfectly clean. Finding out how is part of the job.

## Requirements
- [ ] **Bronze:** raw listens as strings + `_source_file` + `_ingested_at`
- [ ] **Silver:** typed, deduplicated listens; invalid rows go to a **quarantine** table (explain your rules)
- [ ] **Gold:** at least 2 tables that answer the questions above
- [ ] **Quality checks:** at least 3, including 1 **reconciliation** check; the pipeline stops if one fails
- [ ] **`run_pipeline()`** runs everything; show that running it twice gives the same numbers
- [ ] A chart or table with your answer, and a `README.md` on GitHub
- [ ] **Stretch:** make it incremental (session 9), or add an API/JSON source

The structure below is a suggestion. Change it as you like.

In [ ]:
# Run me first: finds the course data, or downloads it when running in Google Colab.
import subprocess
from pathlib import Path

COURSE_REPO = "https://github.com/alinafazliu/data-engineering-modul-1.git"


def find_data():
    for folder in [Path.cwd(), *Path.cwd().parents, Path("/content/course")]:
        for candidate in [folder / "data", folder / "month_course" / "data"]:
            if (candidate / "reference" / "tracks.csv").exists():
                return candidate


DATA = find_data()
if DATA is None:
    subprocess.run(["git", "clone", "--depth", "1", "--quiet", COURSE_REPO, "/content/course"], check=True)
    DATA = find_data()
print("Course data:", DATA)

In [ ]:
# Start Spark. Colab already has PySpark; anywhere else we install it first.
import importlib.util
import os
import subprocess
import sys

if importlib.util.find_spec("pyspark") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "pyspark>=3.5,<3.6"], check=True)
os.environ["PYSPARK_PYTHON"] = sys.executable  # Spark's workers must use the same Python as the notebook

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("wavely")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.ansi.enabled", "false")  # same behaviour on Spark 3.5 and 4.x: bad casts give null
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "is ready")

In [ ]:
SRC = DATA / "capstone" / "podcasts"
LAKE = DATA.parent / "capstone_lake" / "podcasts"


def check(name, bad_rows):
    n_bad = bad_rows.count()
    print(f"{'PASS' if n_bad == 0 else 'FAIL'}  {name}  ({n_bad} bad rows)")
    return n_bad == 0


def read_ref(name):
    return spark.read.csv(str(SRC / f"{name}.csv"), header=True, inferSchema=True)

## 1. Explore
Look at the data first. What is wrong with it?

In [ ]:
# YOUR CODE HERE

## 2. Bronze

In [ ]:
def build_bronze():
    # YOUR CODE HERE
    ...

## 3. Silver
Decide your rules. Write them down here:
- ...

In [ ]:
def build_silver():
    # YOUR CODE HERE
    ...

## 4. Gold

In [ ]:
def build_gold():
    # YOUR CODE HERE
    ...

## 5. Quality checks

In [ ]:
def run_checks():
    # YOUR CODE HERE
    ...

## 6. Run it (twice)

In [ ]:
def run_pipeline():
    build_bronze()
    build_silver()
    build_gold()
    run_checks()
    return spark.read.parquet(str(LAKE / "gold" / "daily_device")).agg(F.sum("listens")).first()[0]


print("Idempotent:", run_pipeline() == run_pipeline())

## 7. Answers

In [ ]:
# YOUR CODE HERE